# 01 — Data Cleaning & Imputation
### Paying for Clean Air? Airbnb Pricing and Urban Air Quality
*University Lab Project*

## Introduction

This notebook walks through the full **data preparation pipeline** for the project:

| Step | Description |
|------|-------------|
| 1 | Load raw CSVs for **30 cities** from `data/raw/` |
| 2 | **Schema standardisation** — rename columns, enforce dtypes, add `city` tag |
| 3 | **Currency conversion** — all prices converted to Indian Rupees (INR) using fixed exchange rates |
| 4 | **Missing value imputation** — median for numeric, mode for categorical, 0-fill for review counts |
| 5 | **Merge** all city frames into a single `merged_listings.csv` saved to `data/processed/` |

> **Scripts used:** `scripts/currency_converter.py`, `scripts/data_merger.py`

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'scripts'))

from currency_converter import CITY_CURRENCY_MAP, EXCHANGE_RATES_TO_INR, convert_to_inr
from data_merger import merge_all_cities, SYNTHETIC_AQI, EXPECTED_COLUMNS

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
print('Libraries loaded ✓')

## 1. Load Raw Data Sample

We peek at `new_york_listings.csv` to understand the raw schema before any transformations are applied.

In [ ]:
raw_path = ROOT / 'data' / 'raw' / 'new_york_listings.csv'
df_raw = pd.read_csv(raw_path)
print(f'Shape: {df_raw.shape}')
display(df_raw.head())
df_raw.info()

## 2. Schema Standardization

`EXPECTED_COLUMNS` defines the canonical column set every city frame must have:

```
id, name, host_id, host_name, neighbourhood, latitude, longitude,
room_type, price, minimum_nights, number_of_reviews, last_review,
reviews_per_month, calculated_host_listings_count, availability_365,
review_scores_rating, city, price_inr, aqi
```

Missing columns are added with `NaN`; extra columns are dropped.

In [ ]:
print('Expected columns:')
print(EXPECTED_COLUMNS)

# Show which columns the raw NY file has vs. what we expect
raw_cols = set(df_raw.columns)
expected_cols = set(EXPECTED_COLUMNS)

print('\n➖ Extra cols (will be dropped):', raw_cols - expected_cols)
print('➕ Missing cols (will be added as NaN):', expected_cols - raw_cols)

## 3. Currency Conversion to INR

Each city is mapped to its local currency via `CITY_CURRENCY_MAP`. Fixed exchange rates (to INR) stored in `EXCHANGE_RATES_TO_INR` are applied. The resulting `price_inr` column is used throughout all analyses.

In [ ]:
# City → currency mapping
currency_df = pd.DataFrame(
    list(CITY_CURRENCY_MAP.items()), columns=['City', 'Currency']
).sort_values('City')
print('City → Currency mapping:')
display(currency_df)

# Exchange rates
rates_df = pd.DataFrame(
    list(EXCHANGE_RATES_TO_INR.items()), columns=['Currency', 'Rate (1 unit → INR)']
).sort_values('Rate (1 unit → INR)', ascending=False)
print('\nExchange rates to INR:')
display(rates_df)

# Demo conversion
demo_usd = 200
demo_inr = convert_to_inr(demo_usd, 'USD')
print(f'\nDemo: {demo_usd} USD = {demo_inr:,.2f} INR')

## 4. Missing Value Analysis

Before imputation we quantify the extent of missingness in the sample city frame.

In [ ]:
df_sample = pd.read_csv(raw_path)

missing = df_sample.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_sample) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
missing_pct[missing_pct > 0].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Missing Values by Column — New York (raw)', fontsize=13)
ax.set_ylabel('Missing %')
ax.set_xlabel('')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 5. Imputation Strategy

| Column | Strategy | Rationale |
|--------|----------|-----------|
| `price` | Median (per city) | Right-skewed; median is robust |
| `reviews_per_month` | Fill `0` | NaN means no reviews yet |
| `last_review` | Fill `'Unknown'` | Categorical date-string |
| `review_scores_rating` | Median (per city) | Only scored listings have it |
| `review_scores_*` | Median (global) | Sub-category scores |
| `calculated_host_listings_count` | Fill `1` | Host has at least this listing |
| `neighbourhood` | Fill `'Unknown'` | Missing geo-label |

In [ ]:
# --- Before imputation ---
numeric_cols = df_sample.select_dtypes(include='number').columns.tolist()
print('=== Before Imputation ===')
print(df_sample[numeric_cols].describe().T[['count','mean','50%','std']])

In [ ]:
# --- Apply simple imputation ---
df_imp = df_sample.copy()

if 'price' in df_imp.columns:
    df_imp['price'] = pd.to_numeric(df_imp['price'].astype(str).str.replace(r'[$,]','',regex=True), errors='coerce')
    df_imp['price'].fillna(df_imp['price'].median(), inplace=True)

if 'reviews_per_month' in df_imp.columns:
    df_imp['reviews_per_month'].fillna(0, inplace=True)

if 'last_review' in df_imp.columns:
    df_imp['last_review'].fillna('Unknown', inplace=True)

if 'review_scores_rating' in df_imp.columns:
    df_imp['review_scores_rating'].fillna(df_imp['review_scores_rating'].median(), inplace=True)

if 'neighbourhood' in df_imp.columns:
    df_imp['neighbourhood'].fillna('Unknown', inplace=True)

print('Imputation applied ✓')

In [ ]:
# --- After imputation ---
print('=== After Imputation ===')
print(df_imp[numeric_cols].describe().T[['count','mean','50%','std']])

In [ ]:
# Rows where price changed due to imputation
if 'price' in df_sample.columns:
    changed = df_sample['price'].isnull().sum()
    print(f'{changed} rows had price imputed with median.')

## 6. Merge All Cities

`merge_all_cities()` iterates over all CSVs in `data/raw/`, applies the schema standardisation + currency conversion + imputation pipeline for each city, appends a synthetic `aqi` column from `SYNTHETIC_AQI`, and concatenates everything into a single DataFrame which it saves to `data/processed/merged_listings.csv`.

In [ ]:
merged = merge_all_cities()
print(f'Merged shape: {merged.shape}')
display(merged.head())
display(merged.describe())

## 7. Data Quality Report

A multi-panel quality overview of the merged dataset.

In [ ]:
# --- Listings per city ---
city_counts = merged['city'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

city_counts.plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Listings per City', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# --- Missing value heatmap (sample columns) ---
key_cols = ['price_inr', 'aqi', 'reviews_per_month', 'review_scores_rating',
            'availability_365', 'minimum_nights', 'latitude', 'longitude']
key_cols = [c for c in key_cols if c in merged.columns]
missing_pivot = merged.groupby('city')[key_cols].apply(lambda x: x.isnull().mean() * 100)
sns.heatmap(missing_pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Missingness % by City & Column', fontsize=13)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# --- Price distribution overview ---
fig, ax = plt.subplots(figsize=(10, 4))
merged['price_inr'].clip(upper=merged['price_inr'].quantile(0.99)).plot(
    kind='hist', bins=60, ax=ax, color='cornflowerblue', edgecolor='white'
)
ax.set_title('Price Distribution (INR) — 99th percentile clipped', fontsize=13)
ax.set_xlabel('Price (INR)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## 8. Save Processed Data

`merge_all_cities()` already writes the merged CSV. Here we verify the file exists and print final summary statistics.

In [ ]:
processed_path = ROOT / 'data' / 'processed' / 'merged_listings.csv'
if processed_path.exists():
    size_mb = processed_path.stat().st_size / 1_048_576
    print(f'✓ {processed_path.name} exists ({size_mb:.2f} MB)')
    final = pd.read_csv(processed_path)
    print(f'  Rows: {len(final):,}  |  Columns: {final.shape[1]}')
    print(f'  Cities: {sorted(final["city"].unique())}')
    display(final.describe())
else:
    print('⚠ File not found — re-run merge_all_cities()')